# Prompt Chaining workflow 

In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict

In [9]:
load_dotenv()

True

In [10]:
# model
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [12]:
def create_outline(state: BlogState) -> BlogState:

    title = state['title']

    # call llm to get outline
    prompt = f'Generate a outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

   # update the outline state
    state['outline'] = outline

    return state

In [13]:
# create blog

def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'write a detailed blog on the title - {title} using the following outline \n {outline}'
    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [11]:
# define the state

class BlogState(TypedDict):

    title: str
    outline: str
    content: str   
    

In [14]:
# create a graph

graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

# compile the graph

workflow = graph.compile()

In [17]:
# final state 
initial_state = {'title': 'Ai impact worldwide'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Ai impact worldwide', 'outline': "# Blog Outline: The Global Impact of AI\n\n## Introduction\n- Brief overview of AI and its rapid development\n- Importance of discussing AI's worldwide impact\n- Purpose of the blog: to explore various dimensions of AI's influence across different sectors and regions\n\n## 1. Understanding AI: A Brief Overview\n- Definition of Artificial Intelligence\n- Types of AI: Narrow AI vs. General AI\n- Key technologies driving AI advancements (e.g., machine learning, natural language processing)\n\n## 2. Economic Impact of AI\n### 2.1 Job Creation and Displacement\n- Overview of jobs created by AI technologies\n- Sectors most affected by automation\n- Strategies for workforce adaptation and reskilling\n\n### 2.2 Productivity and Efficiency\n- How AI enhances productivity in various industries\n- Case studies of businesses leveraging AI for operational efficiency\n\n### 2.3 Economic Growth and Innovation\n- AI's role in driving economic growth in diff